In [2]:
import os, sys
from pathlib import Path
import numpy as np
import torch
import imageio.v2 as imageio
from tqdm import tqdm

# --- Cosmos Tokenizer import ---
sys.path.append("/root/work/src/Cosmos-Tokenizer/")
from cosmos_tokenizer.video_lib import CausalVideoTokenizer

# ----------------------------
# 設定
# ----------------------------
root = "/root/work/data/raw/train_v2.0"
model_name = "Cosmos-Tokenizer-DV8x8x8"
dec_ckpt = f"/root/work/src/Cosmos-Tokenizer/pretrained_ckpts/{model_name}/decoder.jit"

out_dir = Path("/root/work/data/outputs/videos")
out_dir.mkdir(parents=True, exist_ok=True)

start_i, end_i = 0, 50          # inclusive
fps = 30                        # GIFのfps
device = "cuda"                 # "cuda" 推奨

# ----------------------------
# Dataset / Decoder
# ----------------------------
# ds = ShardedBlockTokenDataset(
#     root=root,
#     output_format="seq2seq",
#     drop_last_incomplete_block=True,
#     device=None,
# )

decoder = CausalVideoTokenizer(checkpoint_dec=dec_ckpt)

# ----------------------------
# decode helper
# ----------------------------
def decode_one(tok_3x32x32: np.ndarray) -> torch.Tensor:
    """return decoded tensor (5D)"""
    t = torch.from_numpy(tok_3x32x32.astype(np.int64, copy=False)).to(device)

    candidates = [
        t.unsqueeze(0),                              # (1,3,32,32)
        t.unsqueeze(0).unsqueeze(1),                 # (1,1,3,32,32)
        t.permute(1,2,0).unsqueeze(0),               # (1,32,32,3)
        t.permute(1,2,0).unsqueeze(0).unsqueeze(1),  # (1,1,32,32,3)
    ]

    last_err = None
    for idxs in candidates:
        try:
            with torch.no_grad():
                return decoder.decode(idxs)
        except Exception as e:
            last_err = e
    raise RuntimeError(f"decode failed for all layouts. last error={repr(last_err)}")

def to_uint8_THWC(x: torch.Tensor) -> np.ndarray:
    """decoded tensor -> uint8 video (T,H,W,3)"""
    x = x.detach().float().cpu()
    if x.ndim != 5:
        raise ValueError(f"decoded tensor must be 5D, got {x.ndim}D")

    # (B,3,T,H,W) or (B,T,3,H,W)
    if x.shape[1] == 3:
        x = x[0].permute(1,2,3,0)   # (T,H,W,3)
    elif x.shape[2] == 3:
        x = x[0].permute(0,2,3,1)   # (T,H,W,3)
    else:
        # grayscale (B,1,T,H,W) など
        x = x[0]
        if x.shape[0] == 1:
            x = x[0][..., None].repeat(1,1,3)
        else:
            raise ValueError(f"unrecognized channel layout: {tuple(x.shape)}")

    vmin, vmax = float(x.min()), float(x.max())
    # [-1,1] or [0,1] を想定
    if vmax <= 1.5 and vmin >= -1.5:
        if vmin < 0:
            x = (x + 1.0) / 2.0
        x = x.clamp(0, 1) * 255.0
    else:
        x = x.clamp(0, 255)

    return x.round().to(torch.uint8).numpy()


In [ ]:

# ----------------------------
# main loop
# ----------------------------
torch.cuda.empty_cache()

for i in tqdm(range(start_i, end_i + 1)):
    out_path = out_dir / f"decoded_sample{i}.gif"
    if out_path.exists():
        continue

    sample = ds[i]
    past_tok = sample["past_frames"]     # (3,32,32)
    fut_tok  = sample["future_frames"]  # (3,32,32)

    # decode past/future separately, then concat in time
    past_vid = decode_one(past_tok)
    fut_vid  = decode_one(fut_tok)

    past_np = to_uint8_THWC(past_vid)
    fut_np  = to_uint8_THWC(fut_vid)

    full_np = np.concatenate([past_np, fut_np], axis=0)  # (T,H,W,3)

    # save gif
    imageio.mimsave(out_path, list(full_np), fps=fps)

    # free GPU memory
    del past_vid, fut_vid
    torch.cuda.empty_cache()

print("Done. Saved to:", out_dir)


In [6]:
pred_dir = Path("/root/work/data/processed/")
out_dir = Path("/root/work/data/processed/")
for i in tqdm(range(start_i, end_i + 1), desc="decode+gif"):
    npy_path = pred_dir / f"decoded_pred_{i:06d}.npy"  # ★あなたのprefixに合わせて変えてOK
    if not npy_path.exists():
        print("missing:", npy_path)
        continue

    tok = np.load(npy_path)  # (3,32,32)
    dec = decode_one(tok)
    vid = to_uint8_THWC(dec)  # (T,H,W,3) uint8

    out_path = out_dir / f"decoded_sample{i}.gif"
    imageio.mimsave(out_path, list(vid), fps=fps)

print("done:", out_dir)

decode+gif: 100%|██████████| 51/51 [00:09<00:00,  5.21it/s]

missing: /root/work/data/processed/decoded_pred_000004.npy
missing: /root/work/data/processed/decoded_pred_000005.npy
missing: /root/work/data/processed/decoded_pred_000006.npy
missing: /root/work/data/processed/decoded_pred_000007.npy
missing: /root/work/data/processed/decoded_pred_000008.npy
missing: /root/work/data/processed/decoded_pred_000009.npy
missing: /root/work/data/processed/decoded_pred_000010.npy
missing: /root/work/data/processed/decoded_pred_000011.npy
missing: /root/work/data/processed/decoded_pred_000012.npy
missing: /root/work/data/processed/decoded_pred_000013.npy
missing: /root/work/data/processed/decoded_pred_000014.npy
missing: /root/work/data/processed/decoded_pred_000015.npy
missing: /root/work/data/processed/decoded_pred_000016.npy
missing: /root/work/data/processed/decoded_pred_000017.npy
missing: /root/work/data/processed/decoded_pred_000018.npy
missing: /root/work/data/processed/decoded_pred_000019.npy
missing: /root/work/data/processed/decoded_pred_000020.n